# Hybrid recommenders
All hybrids combine collaborative filtering with the TF-IDF content-based recommender.

- **Parallel:** reciprocal rank fusion of the two top-10 lists
- **Switching:** content-based for sparse users, collaborative filtering for the rest
- **Sequential:** content-based recommendations become synthetic ratings for sparse users, then KNN/SVD is trained on the enriched data

In [1]:
import sys
sys.path.append("..")

from hybrid_recommenders.data import load_train_and_test_ratings

DATA_FOLDER = "../data"
train_df, test_df = load_train_and_test_ratings(f"{DATA_FOLDER}/train_video_games.parquet",
                                                f"{DATA_FOLDER}/test_video_games.parquet")

import pandas as pd

from hybrid_recommenders.content_based import fit_tfidf, recommend_by_profile_similarity
from hybrid_recommenders.data import load_item_metadata
from hybrid_recommenders.evaluation import compare_models
from hybrid_recommenders.hybrids import add_content_based_ratings_for_sparse_users
from hybrid_recommenders.text_preprocessing import build_item_corpus
from run_experiments import recommendations_from_all_models

item_metadata_df = load_item_metadata(f"{DATA_FOLDER}/meta_video_games.parquet", train_df, test_df)

A user is sparse if they have at most 16 ratings, which is the median:

In [2]:
train_df.groupby("user_id")["item_id"].count().describe()

count    1389.000000
mean       19.136069
std        10.197237
min        12.000000
25%        13.000000
50%        16.000000
75%        21.000000
max       114.000000
Name: item_id, dtype: float64

Ratings per user after adding synthetic ratings to the sparse users:

In [3]:
tfidf_matrix, _ = fit_tfidf(build_item_corpus(item_metadata_df))
tfidf_recommendations = recommend_by_profile_similarity(tfidf_matrix, item_metadata_df["item_id"].tolist(), train_df)

enriched_train_df = add_content_based_ratings_for_sparse_users(train_df, tfidf_recommendations)
enriched_train_df.groupby("user_id")["item_id"].count().describe()

count    1389.000000
mean       24.614831
std         8.275904
min        17.000000
25%        22.000000
50%        23.000000
75%        25.000000
max       114.000000
Name: item_id, dtype: float64

## Final comparison

In [4]:
recommendations_per_model = recommendations_from_all_models(train_df, test_df, item_metadata_df, include_embeddings=True)
compare_models(recommendations_per_model, test_df)

/opt/anaconda3/envs/hybrid-recsys/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8154.70it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when lo

,HR,P@10,MAP@10,MRR@10,Coverage
TopPop,0.212,0.024,0.017,0.066,0.011
KNN,0.037,0.004,0.002,0.010,0.510
SVD,0.056,0.006,0.005,0.019,0.077
CB TF-IDF,0.205,0.023,0.020,0.075,0.613
CB GloVe,0.092,0.010,0.009,0.035,0.227
CB BERT,0.084,0.009,0.006,0.030,0.264
CB RoBERTa,0.068,0.007,0.006,0.024,0.231
Parallel RRF (KNN+CB),0.133,0.014,0.010,0.038,0.679
Parallel RRF (SVD+CB),0.143,0.015,0.012,0.047,0.521
Switching (KNN+CB),0.117,0.013,0.012,0.039,0.735
